In [1]:
import pandas as pd
import sqlite3 as sql

In [2]:
path_vendas = './datawarehouse/csv/fato_vendas.csv'
df_vendas = pd.read_csv(path_vendas)
df_vendas.head()

,id_venda,id_cliente,id_vendedor,id_produto,id_tempo,quantidade,preco_unitario,desconto,valor_total,canal,forma_pagamento,hora_venda
0,1,NaN,13,48,20260403,1,266.71,0.0,266.71,Física,Cartão de Débito,12:15
1,2,25.0,24,134,20261024,4,129.26,0.0,517.04,Física,Cartão de Débito,21:45
2,3,23.0,3,119,20260529,2,179.32,0.0,358.64,Física,Crediário,19:45
3,4,312.0,3,32,20260808,5,64.41,16.1,305.95,Física,Crediário,09:30
4,5,238.0,18,32,20260626,5,64.41,0.0,322.05,Física,PIX,19:45


In [3]:
def mapper(df):
    pares = []
    for idx, row in df.iterrows():
        canal = row['canal']
        valor = row['valor_total']
        if pd.notna(canal) and pd.notna(valor):
            pares.append((canal, valor))
    return pares

In [4]:
def shuffle(mapped_data):
    agrupado = {}
    for chave, valor in mapped_data:
        if chave not in agrupado:
            agrupado[chave] = []
        agrupado[chave].append(valor)
    return agrupado

In [5]:
def reducer(agrupado):
    return {
        chave: round(sum(valores), 2)
        for chave, valores in agrupado.items()
    }

In [6]:
mapeado = mapper(df_vendas)
agrupado = shuffle(mapeado)
resultado_mapreduce = reducer(agrupado)

print("[MapReduce] Faturamento total por canal de venda:")
print(resultado_mapreduce)

[MapReduce] Faturamento total por canal de venda:
{'Física': 4086330.96, 'Online': 4056372.76}


In [7]:
jobs = []

def enviar_job(nome, funcao):
    print(f"[YARN] Iniciando job {nome}")
    resultado = funcao()
    jobs.append({
        'nome': nome,
        'status': 'finalizado',
        'resultado': resultado
    })
    return resultado

def job_faturamento_por_forma_pagamento():
    df = df_vendas[['forma_pagamento', 'valor_total']]
    resultado = (
        df.groupby('forma_pagamento')['valor_total']
        .sum()
        .round(2)
        .to_dict()
    )
    return resultado

In [8]:
resultado_yarn = enviar_job(
    "Faturamento por Forma de Pagamento",
    job_faturamento_por_forma_pagamento
)

print("\nResultado da JOB:")
for forma, total in resultado_yarn.items():
    print(f"Forma de pagamento '{forma}': R$ {total:,.2f}")

[YARN] Iniciando job Faturamento por Forma de Pagamento

Resultado da JOB:
Forma de pagamento 'Cartão de Crédito': R$ 2,204,259.75
Forma de pagamento 'Cartão de Débito': R$ 2,262,851.89
Forma de pagamento 'Crediário': R$ 830,456.42
Forma de pagamento 'Dinheiro': R$ 1,013,284.76
Forma de pagamento 'PIX': R$ 1,831,850.90


In [9]:
print("\n[YARN] Histórico de jobs:")
for job in jobs:
    print(f"{job['nome']} - {job['status']}")


[YARN] Histórico de jobs:
Faturamento por Forma de Pagamento - finalizado


In [10]:
conn = sql.connect(":memory:")

df_vendas[['id_venda', 'id_cliente', 'valor_total', 'canal', 'forma_pagamento']].to_sql(
    "vendas",
    conn,
    index=False
)

14806

In [11]:
df_sql = pd.read_sql(
    "SELECT canal, COUNT(*) as total_vendas, SUM(valor_total) as faturamento "
    "FROM vendas "
    "WHERE valor_total > 200 "
    "GROUP BY canal",
    conn
)

print(df_sql)

    canal  total_vendas  faturamento
0  Física          5840   3897918.56
1  Online          5812   3867007.14


In [12]:
df_sql.to_csv('./sqoop/vendas_alto_valor.csv', index=False)

In [13]:
df_lido = pd.read_csv('./sqoop/vendas_alto_valor.csv')
df_lido

,canal,total_vendas,faturamento
0,Física,5840,3897918.56
1,Online,5812,3867007.14
